In [ ]:
# @title 1. Install System Dependencies & Clone ComfyUI
import os
from IPython.display import clear_output

# 1. Install system tools
!apt-get update -y && apt-get install -y ffmpeg libgl1-mesa-glx aria2

# 2. Clone ComfyUI
if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
    print("✅ ComfyUI cloned.")
else:
    print("ℹ️ ComfyUI already exists.")

# 3. Clone ComfyUI Manager (Essential for installing missing nodes)
if not os.path.exists("ComfyUI/custom_nodes/ComfyUI-Manager"):
    %cd ComfyUI/custom_nodes
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git
    %cd ../..
    print("✅ ComfyUI Manager installed.")

# 4. Install Python Requirements
%cd ComfyUI
!pip install -r requirements.txt
!pip install pyngrok  # Needed for the tunnel later

print("✅ Installation complete.")
clear_output()

In [ ]:
# @title 2. Apply Storage Hack (Move Models to /tmp)
import os
import shutil

# Define paths
BASE_DIR = "/kaggle/working/ComfyUI"
TEMP_DIR = "/tmp/ComfyUI_Storage" # /tmp usually has ~Disk space on Kaggle

# Create temp directory
os.makedirs(TEMP_DIR, exist_ok=True)

# Function to move folder to temp and symlink back
def setup_storage_link(folder_name):
    src = os.path.join(BASE_DIR, folder_name)
    dst = os.path.join(TEMP_DIR, folder_name)
    
    # If the link already exists, skip
    if os.path.islink(src):
        print(f"ℹ️ {folder_name} is already linked.")
        return

    # If the source folder exists (real folder), move it to temp
    if os.path.exists(src):
        # Create destination if missing
        if not os.path.exists(dst):
            print(f"📦 Moving existing {folder_name} to {dst}...")
            shutil.move(src, dst)
        else:
            # If both exist, merge/delete source (safe for empty default folders)
            print(f"⚠️ {dst} exists. Merging/Replacing {src}...")
            shutil.rmtree(src)
    
    # Create the symlink
    if not os.path.exists(src):
        os.symlink(dst, src)
        print(f"🔗 Linked {src} -> {dst}")

# Apply to heavy folders
setup_storage_link("models")
setup_storage_link("input")
setup_storage_link("output")

print("✅ Storage hack applied successfully.")

In [ ]:
import os
os.environ["HF_TOKEN"] = "YOUR_TOKEN"  # @param {type:"string"}

In [ ]:
!apt-get update -qq
!apt-get install -y -qq aria2
!aria2c --version

In [ ]:
import os
import subprocess
import struct
import json

# =========================================================
# CONFIG
# =========================================================
BASE = "/kaggle/working/ComfyUI/models"

DIRS = {
    "diffusion_models": os.path.join(BASE, "diffusion_models"),
    "text_encoders": os.path.join(BASE, "text_encoders"),
    "vae": os.path.join(BASE, "vae"),
}

for folder in DIRS.values():
    os.makedirs(folder, exist_ok=True)

# =========================================================
# MODEL LINKS — THEO ĐÚNG ẢNH ANH GỬI
# Dùng /resolve/main/ để tải file thực, không tải trang HTML
# =========================================================
REPO = "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main"

downloads = [
    # Diffusion models
    (
        f"{REPO}/diffusion_models/qwen_image_2.1_bf16.safetensors",
        DIRS["diffusion_models"],
        "qwen_image_2.1_bf16.safetensors",
    ),
    (
        f"{REPO}/diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
        DIRS["diffusion_models"],
        "qwen_image_2.1_int8_convrot.safetensors",
    ),

    # Text encoders — tên chính xác theo ảnh
    (
        f"{REPO}/text_encoders/qwen3vl_8b_bf16.safetensors",
        DIRS["text_encoders"],
        "qwen3vl_8b_bf16.safetensors",
    ),
    (
        f"{REPO}/text_encoders/qwen3vl_8b_int8_convrot.safetensors",
        DIRS["text_encoders"],
        "qwen3vl_8b_int8_convrot.safetensors",
    ),

    # VAE
    (
        f"{REPO}/vae/qwen_image_2.1_vae_bf16.safetensors",
        DIRS["vae"],
        "qwen_image_2.1_vae_bf16.safetensors",
    ),
]

# =========================================================
# VALIDATE SAFETENSORS
# =========================================================
def is_valid_safetensors(path):
    try:
        size = os.path.getsize(path)

        # File phải có ít nhất 8 bytes header + nội dung
        if size < 16:
            return False, "File quá nhỏ"

        with open(path, "rb") as f:
            raw = f.read(8)

            if len(raw) != 8:
                return False, "Không đọc được header"

            header_size = struct.unpack("<Q", raw)[0]

            # Header phải nằm trong phạm vi file và có kích thước hợp lý
            if header_size <= 1 or header_size > size - 8:
                return False, f"Header không hợp lệ: {header_size:,} bytes"

            if header_size > 100 * 1024 * 1024:
                return False, f"Header quá lớn: {header_size:,} bytes"

            header = f.read(header_size)

            # Safetensors header phải là JSON
            json.loads(header.decode("utf-8"))

        return True, "Header hợp lệ"

    except Exception as e:
        return False, str(e)


# =========================================================
# DOWNLOAD
# =========================================================
def download_model(url, folder, filename):
    path = os.path.join(folder, filename)

    # Bỏ qua file đã tải và vượt qua kiểm tra header
    if os.path.exists(path):
        valid, message = is_valid_safetensors(path)

        if valid:
            print(f"✅ Đã có model hợp lệ: {filename}")
            print(f"   Size: {os.path.getsize(path) / (1024**3):.2f} GB")
            return

        print(f"⚠️ File hiện tại không hợp lệ: {filename}")
        print(f"   Lý do: {message}")
        print("   Xóa file lỗi để tải lại...")
        os.remove(path)

    print(f"\n⬇️ Đang tải: {filename}")

    # aria2c ghi trực tiếp vào đúng thư mục
    cmd = [
        "aria2c",
        "-x", "8",
        "-s", "8",
        "-c",
        "--allow-overwrite=true",
        "--auto-file-renaming=false",
        "--check-integrity=true",
        "-o", filename,
        "-d", folder,
        url,
    ]

    result = subprocess.run(cmd)

    if result.returncode != 0:
        raise RuntimeError(
            f"Tải thất bại: {filename}. "
            "Kiểm tra URL, quyền truy cập và dung lượng Kaggle."
        )

    if not os.path.exists(path):
        raise RuntimeError(f"Không tìm thấy file sau khi tải: {path}")

    valid, message = is_valid_safetensors(path)

    if not valid:
        # Không giữ lại file lỗi để tránh ComfyUI tải nhầm
        os.remove(path)
        raise RuntimeError(
            f"File tải về không phải safetensors hợp lệ: {filename}. "
            f"Chi tiết: {message}"
        )

    print(f"✅ Tải và kiểm tra thành công: {filename}")
    print(f"   Size: {os.path.getsize(path) / (1024**3):.2f} GB")


# =========================================================
# RUN DOWNLOADS
# =========================================================
for url, folder, filename in downloads:
    download_model(url, folder, filename)

# =========================================================
# SHOW MODEL FILES
# =========================================================
print("\n" + "=" * 60)
print("DANH SÁCH MODEL SAU KHI TẢI")
print("=" * 60)

for name, folder in DIRS.items():
    print(f"\n📁 {name}/")

    for filename in sorted(os.listdir(folder)):
        path = os.path.join(folder, filename)

        if os.path.isfile(path):
            size_gb = os.path.getsize(path) / (1024**3)
            print(f"   {filename} ({size_gb:.2f} GB)")

print("\nHoàn tất kiểm tra các model đã tải.")

In [ ]:
# @title 3. Run ComfyUI + Pinggy tunnel (no ngrok)
import subprocess, threading, re, time, os

PORT = 8188
COMFY_DIR = "/kaggle/working/ComfyUI"

# Make sure ssh exists (Pinggy uses ssh reverse tunneling)
!apt-get update -y && apt-get install -y openssh-client

# Clean old processes
!pkill -f "main.py" || true
!pkill -f "a.pinggy.io" || true

def run_comfyui():
    cmd = f"python main.py --listen 0.0.0.0 --port {PORT}"
    subprocess.run(cmd, shell=True, cwd=COMFY_DIR)

def run_pinggy():
    cmd = (
        f"ssh -T -o StrictHostKeyChecking=no "
        f"-o ServerAliveInterval=30 -o ServerAliveCountMax=3 "
        f"-o BatchMode=no "
        f"-p 443 -R0:127.0.0.1:{PORT} a.pinggy.io"
    )

    p = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    url_re = re.compile(r"(https?://\S+)")
    printed = False

    for line in p.stdout:
        if printed:
            continue

        m = url_re.search(line)
        if not m:
            continue

        url = m.group(1)

        # ❌ skip dashboard
        if "dashboard.pinggy.io" in url:
            continue

        # ✅ real tunnel domains
        if any(x in url for x in ("pinggy.link", "pinggy.live", "pinggy.io")):
            print("\n🚀 ComfyUI Public URL:", url, "\n")
            printed = True



t1 = threading.Thread(target=run_comfyui, daemon=True)
t2 = threading.Thread(target=run_pinggy, daemon=True)
t1.start()
time.sleep(3)  # give ComfyUI a moment to bind the port
t2.start()

print("⏳ Waiting for Pinggy to print the public URL...")


In [ ]:
import requests

try:
    r = requests.get("http://127.0.0.1:8188", timeout=5)
    print("✅ ComfyUI đang phản hồi:", r.status_code)
except Exception as e:
    print("❌ ComfyUI chưa phản hồi:", e)

In [ ]:
import subprocess
import re
import sys

PORT = 8188

cmd = [
    "ssh",
    "-T",
    "-o", "StrictHostKeyChecking=no",
    "-o", "ServerAliveInterval=30",
    "-o", "ServerAliveCountMax=3",
    "-o", "ExitOnForwardFailure=yes",
    "-o", "BatchMode=yes",
    "-p", "443",
    "-R", f"0:127.0.0.1:{PORT}",
    "a.pinggy.io"
]

print("⏳ Đang kết nối Pinggy...", flush=True)

p = subprocess.Popen(
    cmd,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for line in p.stdout:
        print(line, end="", flush=True)

        if re.search(r"https?://[^\s]+", line):
            urls = re.findall(r"https?://[^\s]+", line)
            for url in urls:
                url = url.rstrip(".,)")
                if "dashboard.pinggy.io" not in url:
                    print("\n🚀 Hãy thử mở link:", url, flush=True)

except KeyboardInterrupt:
    p.terminate()